# A_03 The dot product

*Course appendix A: the toolbox (our own, not the book's appendix A). Dip in whenever you meet a tool you don't know yet.*

**How to use this notebook:** click a cell and press Shift+Enter to run just that cell.
Before each code cell, write down what you think it will do. Then run it and compare.

Run the cells in order, from the top. **A cell can use what earlier cells made**, so if you skip ahead you'll see `NameError: name '…' is not defined`. If that happens, run the cells above it, or just click Run All.

> **Where this fits**
>
> ```
> Odyssey text → numbers → [model reads] → model learns → model writes → MCP tool
> ```
>
> You are here: **model reads**
>
> Project files this connects to:
>
> - `src/homer_ai/model.py`
>
> **This notebook:** One small piece of maths sits at the heart of attention (chapter 3) and of `@` in `model.py`. Once you know it, the attention code stops looking like magic.
>
> For the big picture, what we're building and why, see the [course map](../00_start_here/00_00_course_map.ipynb).

## 1. What you'll learn

- How multiplying two lists of numbers gives a similarity score, shown with movies
- What a dot product is, and how to draw it as arrows
- Why we sometimes divide by the lengths (cosine similarity)
- How matrix multiplication is many dot products at once
- Where our model uses it: `Q @ K.T`

## 2. Words you'll meet

| Word | Meaning in plain English | Example |
|---|---|---|
| **vector** | a list of numbers; drawn as an arrow from the centre of a graph | `[5, -4]` |
| **dot product** | multiply two vectors number by number, then add the results | `[1, 2]·[3, 4] = 1×3 + 2×4 = 11` |
| **length (of a vector)** | how long its arrow is: square every number, add them up, take the square root | `[3, 4]` has length 5 |
| **cosine similarity** | the dot product divided by both lengths, so it only compares direction; always between -1 and 1 | `1` = same way, `-1` = opposite |
| **matrix** | a table of numbers: rows and columns | a `2 × 3` table |
| **matrix multiplication** | every row of one matrix dotted with every column of another | `A @ B` |
| **transpose** | flip a matrix so its rows become columns | `K.T` |
| **query and key** | two vectors the model makes for every character: what it's looking for, and what it offers | `Q` and `K` in `model.py` |
| **heatmap** | a table drawn as coloured boxes, so big and small numbers are easy to see | the `Q @ K.T` picture below |

## 3. The idea

### ❓ How can a few numbers say whether two things are similar?

Let's start with movies. Rate each one on two traits, [Action, Romance], from -5 (hate) to +5 (love). Alex loves action and hates romance, so **Alex = [5, -4]**.

To score a movie for Alex, multiply the numbers that belong together (action with action, romance with romance), then add:

| Movie | Vector [Action, Romance] | The maths | Score | Meaning |
|---|---|---|---|---|
| Die Hard | [5, -3] | (5 × 5) + (-4 × -3) = 25 + 12 | **37** | very similar: both love action, both dislike romance |
| The Notebook | [-4, 5] | (5 × -4) + (-4 × 5) = -20 + -20 | **-40** | opposite: the signs clash |
| A cooking show | [0, 0] | (5 × 0) + (-4 × 0) = 0 + 0 | **0** | unrelated: no shared traits |

👉 **Predict first:** will the code below print the same three scores?

In [ ]:
alex = [5, -4]
movies = {"Die Hard": [5, -3], "The Notebook": [-4, 5], "A cooking show": [0, 0]}

for name, v in movies.items():
    parts = [f"({a} × {b})" for a, b in zip(alex, v, strict=True)]
    score = sum(a * b for a, b in zip(alex, v, strict=True))
    print(f"{name:<15}{v}  {' + '.join(parts)} = {score}")

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

`37`, `-40` and `0`, the same as the table. `zip` walks along two lists side by side, and `strict=True` means "they must be the same length". This multiply-then-add is the **dot product**, written `a · b`.

</details>

### ❓ Why does multiplying measure similarity?

- **Matching signs boost the score.** `+ × +` and `- × -` are both positive. So two people who both *dislike* romance get a boost, just like two people who both love action.
- **Clashing signs lower the score.** `+ × -` is negative: one loves what the other hates.
- **Zeros cancel.** If one side doesn't care (0), that trait adds nothing, whatever the other side feels.

### ❓ Can I see it?

Yes. A **vector** is a list of numbers. With two numbers we can draw it as an arrow from the centre `(0, 0)`: the first number says how far right, the second how far up. Here the right-hand direction is Action and the up direction is Romance.

👉 **Predict first:** which movie's arrow will point the same way as Alex's?

In [ ]:
import matplotlib.pyplot as plt

from homer_ai.studio_style import (
    ACCENT,
    BAD,
    GOOD,
    MUTED,
    studio_axes,
    studio_figure,
)

alex = [5, -4]
arrows = [
    # (name, vector, colour)
    ("Alex", alex, ACCENT),
    ("Die Hard", [5, -3], GOOD),
    ("The Notebook", [-4, 5], BAD),
]

fig = studio_figure("Films as arrows", "Same direction, high score")
ax = studio_axes(fig, (0.1, 0.1, 0.55, 0.7))
for _, v, colour in arrows:
    style = {"arrowstyle": "-|>", "color": colour, "lw": 3, "mutation_scale": 18}
    ax.annotate(
        "", xy=v, xytext=(0, 0),
        arrowprops=style,
    )
ax.axhline(0, color=MUTED, linestyle="--", linewidth=0.8)
ax.axvline(0, color=MUTED, linestyle="--", linewidth=0.8)
ax.set(xlim=(-6, 6), ylim=(-6, 6), aspect="equal")
ax.set_xlabel("Action  (-5 hate, +5 love)")
ax.set_ylabel("Romance  (-5 hate, +5 love)")

# the legend: coloured text beside the plot, with the scores worked out by code
y = 0.72
for name, v, colour in arrows:
    score = sum(a * b for a, b in zip(alex, v, strict=True))
    fig.text(0.7, y, name, color=colour, fontsize=14, fontweight="bold")
    fig.text(0.7, y - 0.05, str(v), color=MUTED, fontsize=12, family="monospace")
    if name != "Alex":
        line = f"Alex · {name} = {score:+d}"
        fig.text(0.7, y - 0.1, line, color=colour, fontsize=11, family="monospace")
    y -= 0.2
plt.show()

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

Alex and Die Hard point almost the same way, and their score is high (`37`). The Notebook points the opposite way, and its score is very negative (`-40`). Two arrows at 90° (right angles, like Action-only `[1, 0]` and Romance-only `[0, 1]`) would score `0`, because they have nothing in common.

</details>

### ❓ One quick catch: does a longer arrow score higher?

Yes, and that can mislead. `[100, 100]` and `[1, 1]` point exactly the same way, but one is much longer.

The fix is to divide the dot product by the two lengths of the arrows. That gives the **cosine similarity**, which only looks at direction: `1` means the same way, `0` right angles, `-1` opposite ways.

👉 **Predict first:** will `[100, 100]·[1, 1]` score more than `[1, 1]·[1, 1]`? And what will their cosine similarities be?

In [ ]:
from math import sqrt


def dot(a, b):
    return sum(x * y for x, y in zip(a, b, strict=True))


def length(a):
    return sqrt(dot(a, a))


def cosine(a, b):
    return dot(a, b) / (length(a) * length(b))


print("dot    [1, 1]·[1, 1]    =", dot([1, 1], [1, 1]))
print("dot    [100, 100]·[1, 1] =", dot([100, 100], [1, 1]))
print("cosine [1, 1]·[1, 1]    =", round(cosine([1, 1], [1, 1]), 3))
print("cosine [100, 100]·[1, 1] =", round(cosine([100, 100], [1, 1]), 3))
print("cosine Alex·Die Hard     =", round(cosine([5, -4], [5, -3]), 3))
print("cosine Alex·The Notebook =", round(cosine([5, -4], [-4, 5]), 3))

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

The longer vector has a much bigger dot product, but both pairs have cosine `1.0`. Alex and Die Hard are close to `1`, and Alex and The Notebook are close to `-1`.

</details>

## 4. Step by step

Each step is one question. Read it, guess what the code will do, run it, then look at what you got.

### ❓ How does PyTorch do it?

`torch.dot` does the whole multiply-and-add in one call. The `@` sign does the same for two vectors. We use PyTorch because the model's numbers are PyTorch tensors.

👉 **Predict first:** the dot product of `[1, 2, 3]` and `[4, 5, 6]` is `1×4 + 2×5 + 3×6`. Will both lines print that?

In [ ]:
import torch

a = torch.tensor([1.0, 2.0, 3.0])
b = torch.tensor([4.0, 5.0, 6.0])
print(torch.dot(a, b))
print(a @ b)

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

`tensor(32.)` twice: `4 + 10 + 18 = 32`.

</details>

### ❓ What's matrix multiplication?

A **matrix** is a table of numbers. Multiplying two matrices with `@` means: take every row of the first, and dot it with every column of the second. Each answer fills one box of the result.

So it's really just many dot products at once.

👉 **Predict first:** what goes in the top-left box of the result?

In [ ]:
A = torch.tensor([[1.0, 2.0], [3.0, 4.0]])
B = torch.tensor([[5.0, 6.0], [7.0, 8.0]])
print(A @ B)

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

The top-left box is `19`: row 1 of `A` `[1, 2]` dotted with column 1 of `B` `[5, 7]`, which is `1×5 + 2×7 = 19`. Every other box is another row-column dot product.

</details>

### ❓ What does `.T` (transpose) do?

**Transpose** flips a matrix so its rows become columns. You need it when you want to dot rows with rows. Flip the second matrix, and its rows become columns.

👉 **Predict first:** what shape does a `2 × 3` matrix have after `.T`?

In [ ]:
M = torch.tensor([[1, 2, 3], [4, 5, 6]])
print(M.shape)
print(M.T)
print(M.T.shape)

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

`(2, 3)` becomes `(3, 2)`. The first row `[1, 2, 3]` became the first column.

</details>

## 5. Full example: from movies to characters, `Q @ K.T`

Now swap movies for characters. In `model.py`, the model gives every character two vectors of 32 numbers each: a **query** ("what am I looking for?", like Alex's tastes) and a **key** ("what do I offer?", like a movie's traits). The dot product of one character's query with another's key says how well those two characters match.

The model wants that score for every pair of characters, so it writes `scores = Q @ K.T`. That's all the dot products at once, one row per query and one column per key.

First, 3 characters with made-up 2-number vectors, so you can see the shape of the result.

In [ ]:
# one row per character; made-up numbers to show the idea
Q = torch.tensor([[1.0, 0.0], [0.0, 1.0], [1.0, 1.0]])  # queries for 3 characters
K = torch.tensor([[1.0, 0.0], [0.0, 1.0], [1.0, 1.0]])  # keys for the same 3 characters

scores = Q @ K.T
print(scores)
print(scores.shape)

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

A `3 × 3` table. The box in row 1, column 3 says how well character 1 "matches" character 3.

</details>

### ❓ What do the real scores look like for `ἄνδρα`?

Here are the trained model's own numbers: the 5 characters of `ἄνδρα`, each with a real 32-number query and key. Green boxes are good matches (positive), pink-red boxes clash (negative), and dark is near zero.

👉 **Predict first:** is the table 5 × 5? Is it symmetrical (the same flipped over the diagonal)?

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch

from homer_ai.lessons import load_homer_model
from homer_ai.studio_style import (
    GOOD_BAD,
    INK,
    MUTED,
    studio_figure,
    text_on,
)

tok, model = load_homer_model()  # yours if you trained one, otherwise models/
if model is None:
    print("no trained model found")
else:

    ids = torch.tensor(tok.encode("ἄνδρα"))
    with torch.no_grad():
        x = model.embedding(ids)
        Q, K = model.query(x), model.key(x)
        scores = Q @ K.T
    print("Q", tuple(Q.shape), " K", tuple(K.shape), " scores", tuple(scores.shape))

    letters = list("ἄνδρα")
    grid = scores.numpy()
    limit = float(np.abs(grid).max())
    fig = studio_figure(
        "Attention scores", "How well each character matches", size=(8, 6.5)
    )
    ax = fig.add_axes((0.14, 0.08, 0.66, 0.68))
    ax.set_facecolor("none")
    ax.imshow(grid, cmap=GOOD_BAD, vmin=-limit, vmax=limit)
    for i in range(5):
        for j in range(5):
            ax.text(
                j, i, f"{grid[i, j]:+.1f}", ha="center", va="center",
                color=text_on(grid[i, j], limit), fontsize=12,
                family="monospace", fontweight="bold",
            )
    ax.set_xticks(range(5), letters, fontsize=16, color=INK)
    ax.set_yticks(range(5), letters, fontsize=16, color=INK)
    ax.xaxis.tick_top()
    ax.tick_params(length=0)
    for side in ax.spines.values():
        side.set_visible(False)
    ax.set_xlabel("key: what the character offers", color=MUTED)
    ax.xaxis.set_label_position("top")
    ax.set_ylabel("query: what it looks for", color=MUTED)
    plt.show()
    print("symmetrical?", bool(np.allclose(grid, grid.T)))

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

A `5 × 5` heatmap: row `i`, column `j` is the dot product of character `i`'s query with character `j`'s key. It isn't symmetrical, because a query and a key are two different vectors, even for the same character. Chapter 3 turns this table into attention.

</details>

## Questions learners ask

These are real questions that came up while we were building this project, with short answers.

### ❓ Why is it called a *dot* product?

Because in maths it's written with a dot between the two vectors: `a · b`.

### ❓ Do the two vectors have to be the same length?

Yes. Every number needs a partner to multiply with. PyTorch stops with an error if the lengths differ.

## 6. Recap

- A dot product multiplies pairs and adds them up. Matching signs boost the score, clashing signs lower it, zeros cancel.
- Arrows pointing the same way score high. Dividing by the lengths gives cosine similarity, which only compares direction.
- `Q @ K.T` in `model.py` is every character's query dotted with every character's key, all at once. That's the heart of attention.

**Next:** [A_04 PyTorch in this repo](A_04_pytorch_in_this_repo.ipynb).